# ECE 343 Lab: Reflection and Refraction in a Simulation (Lesson 41)

<a href="https://colab.research.google.com/github/livethisdream/ece343/blob/main/book/extras/labs/ECE343_Lab_L41_Incidence.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Team members:**

**Documentation:**

This lab solves Maxwell's curl equations on a grid, step by step in time, and
measures what Lessons 38 through 40 derived. The method is the
**finite-difference time-domain (FDTD)** method, which engineers use to design
antennas, radomes, and optical coatings. Part A sends a pulse along one axis
into a dielectric and measures $\Gamma$ and $\tau$ (Lesson 38), then designs a
quarter-wave anti-reflection layer. Part B sends a beam at an angle into
acrylic and measures Snell's law (Lesson 39), the Brewster angle, and total
internal reflection (Lesson 40).

**Teams.** Work in a team of two or three. Each team submits one report.

**Deliverable.** Submit this notebook as a PDF on Gradescope, run top to
bottom with every code cell complete and every **Answer** cell filled in. Put
every member's name in the cell above, and write a Documentation line
(**None** if you did not collaborate outside your team).

**Running it.** Open it in Google Colab with the badge above, or run it
locally with Python 3, NumPy, and Matplotlib. Run the cells in order. A cell
marked `TODO` is yours to finish; the rest are given. Part B's simulations
take about three minutes in all.

**Reference.** Ellingson, *Electromagnetics*, Vol. 1, Chapters 8 and 9, and
Vol. 2, Chapter 5; the Lesson 33 through 40 pages on the course site.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

C0 = 299_792_458.0                 # m/s
EPS0 = 8.8541878128e-12            # F/m
MU0 = 4e-7 * np.pi                 # H/m
ETA0 = np.sqrt(MU0 / EPS0)         # intrinsic impedance of free space, ohms
print(f"eta0 = {ETA0:.1f} ohms")

---
## Part A: a pulse in one dimension

### The update equations

A plane wave traveling along $z$ with $\mathbf{E} = \hat{\mathbf{x}}E_x$ and
$\mathbf{H} = \hat{\mathbf{y}}H_y$ depends only on $z$ and $t$. In a
source-free, lossless medium, the $y$ component of Faraday's law and the $x$
component of the Ampere–Maxwell law (both from Lesson 33) are

$$\frac{\partial H_y}{\partial t} = -\frac{1}{\mu}\frac{\partial E_x}{\partial z}, \qquad \frac{\partial E_x}{\partial t} = -\frac{1}{\epsilon}\frac{\partial H_y}{\partial z}.$$

The grid stores $E_x$ at $z = k\Delta z$ and $H_y$ halfway between, at
$z = (k + \tfrac{1}{2})\Delta z$, and it stores them at alternating half time
steps. This staggered arrangement is the **Yee grid**. Replacing each
derivative with a centered difference gives

$$\frac{H_y\big|^{n+1/2}_{k+1/2} - H_y\big|^{n-1/2}_{k+1/2}}{\Delta t} = -\frac{1}{\mu}\ \frac{E_x\big|^{n}_{k+1} - E_x\big|^{n}_{k}}{\Delta z},$$

$$\frac{E_x\big|^{n+1}_{k} - E_x\big|^{n}_{k}}{\Delta t} = -\frac{1}{\epsilon}\ \frac{H_y\big|^{n+1/2}_{k+1/2} - H_y\big|^{n+1/2}_{k-1/2}}{\Delta z},$$

and solving each for the newest value gives the update equations the code runs:

$$H_y\big|^{n+1/2}_{k+1/2} = H_y\big|^{n-1/2}_{k+1/2} - \frac{\Delta t}{\mu\Delta z}\left(E_x\big|^{n}_{k+1} - E_x\big|^{n}_{k}\right),$$

$$E_x\big|^{n+1}_{k} = E_x\big|^{n}_{k} - \frac{\Delta t}{\epsilon\Delta z}\left(H_y\big|^{n+1/2}_{k+1/2} - H_y\big|^{n+1/2}_{k-1/2}\right).$$

**Time step.** We substitute a wave, $E_x = E_0e^{j(\omega n\Delta t - \beta k\Delta z)}$
and $H_y = H_0e^{j(\omega n\Delta t - \beta k\Delta z)}$, into the two updates.
A centered time difference of $e^{j\omega n\Delta t}$ is
$(2j/\Delta t)\sin(\omega\Delta t/2)$ times the wave at the half step, and a
centered space difference of $e^{-j\beta k\Delta z}$ is
$-(2j/\Delta z)\sin(\beta\Delta z/2)$ times the wave at the half node. The two
updates become

$$\epsilon\ \frac{2}{\Delta t}\sin\frac{\omega\Delta t}{2}\ E_0 = \frac{2}{\Delta z}\sin\frac{\beta\Delta z}{2}\ H_0, \qquad \mu\ \frac{2}{\Delta t}\sin\frac{\omega\Delta t}{2}\ H_0 = \frac{2}{\Delta z}\sin\frac{\beta\Delta z}{2}\ E_0.$$

Multiplying them eliminates $E_0H_0$:

$$\mu\epsilon\left(\frac{2}{\Delta t}\right)^2\sin^2\frac{\omega\Delta t}{2} = \left(\frac{2}{\Delta z}\right)^2\sin^2\frac{\beta\Delta z}{2},$$

$$\sin\frac{\omega\Delta t}{2} = S\sin\frac{\beta\Delta z}{2}, \qquad S = \frac{v_p\Delta t}{\Delta z}, \qquad v_p = \frac{1}{\sqrt{\mu\epsilon}}.$$

This is the grid's **dispersion relation**. For $S > 1$ the shortest waves on
the grid, $\beta\Delta z = \pi$, need $\sin(\omega\Delta t/2) > 1$, so $\omega$
is complex and they grow without bound. The **Courant condition** $S \le 1$
keeps the simulation stable. At $S = 1$ the relation reduces to
$\omega = v_p\beta$ and the grid has no dispersion at all; we run air at
$S = 1$, so the dielectric runs at $S = 1/n$.

**Boundaries.** The grid ends, and a wave that reaches the end must leave
without reflecting. At the left end the wave travels toward $-z$, so
$E_x = f(t + z/v_p)$, which obeys the one-way equation

$$\frac{\partial E_x}{\partial z} - \frac{1}{v_p}\frac{\partial E_x}{\partial t} = 0.$$

We write it with centered differences at $z = \Delta z/2$ and
$t = (n + \tfrac{1}{2})\Delta t$, averaging the two nodes and the two time
steps:

$$\frac{E_x\big|^{n+1}_{1} + E_x\big|^{n}_{1} - E_x\big|^{n+1}_{0} - E_x\big|^{n}_{0}}{2\Delta z} = \frac{E_x\big|^{n+1}_{0} + E_x\big|^{n+1}_{1} - E_x\big|^{n}_{0} - E_x\big|^{n}_{1}}{2v_p\Delta t}.$$

Multiplying by $2v_p\Delta t\Delta z$ and collecting the newest end value on
the left gives

$$(v_p\Delta t + \Delta z)\ E_x\big|^{n+1}_{0} = (v_p\Delta t + \Delta z)\ E_x\big|^{n}_{1} + (v_p\Delta t - \Delta z)\left(E_x\big|^{n+1}_{1} - E_x\big|^{n}_{0}\right),$$

and dividing by $v_p\Delta t + \Delta z$ gives the first-order **Mur boundary**

$$E_x\big|^{n+1}_{0} = E_x\big|^{n}_{1} + \frac{v_p\Delta t - \Delta z}{v_p\Delta t + \Delta z}\left(E_x\big|^{n+1}_{1} - E_x\big|^{n}_{0}\right).$$

The right end is its mirror image.

**The node on the boundary.** When a material boundary falls on an $E_x$ node,
the cell around that node, from $k\Delta z - \Delta z/2$ to
$k\Delta z + \Delta z/2$, is half in each medium. $E_x$ is tangential to the
boundary, so it is continuous (Lesson 24) and takes one value across the cell.
Integrating $\epsilon\ \partial E_x/\partial t = -\partial H_y/\partial z$ over
the cell gives

$$\left(\epsilon_1\frac{\Delta z}{2} + \epsilon_2\frac{\Delta z}{2}\right)\frac{\partial E_x}{\partial t} = -\left(H_y\big|_{k+1/2} - H_y\big|_{k-1/2}\right),$$

so that node takes the average permittivity $(\epsilon_1 + \epsilon_2)/2$.

In [ ]:
F0_1D = 1e9                         # design frequency, Hz
LAM0_1D = C0 / F0_1D                # 0.30 m
DZ = LAM0_1D / 80                   # 80 cells per free-space wavelength
DT = DZ / C0                        # S = 1 in air
NZ = 1200
KS, KP, KB, KT = 50, 150, 400, 550  # source, probe, boundary, transmitted probe


def fdtd1d(eps_r, nsteps=4000, pec_from=None):
    # eps_r: relative permittivity at each E node. pec_from: E = 0 from that node on.
    Ex = np.zeros(NZ)
    Hy = np.zeros(NZ - 1)
    ce = DT / (EPS0 * eps_r * DZ)
    ch = DT / (MU0 * DZ)
    vL, vR = C0 / np.sqrt(eps_r[0]), C0 / np.sqrt(eps_r[-1])
    mL = (vL * DT - DZ) / (vL * DT + DZ)
    mR = (vR * DT - DZ) / (vR * DT + DZ)
    tw, t0 = 0.5 / F0_1D, 2.0 / F0_1D           # Gaussian envelope: width and center
    rec = {"p": np.zeros(nsteps), "t": np.zeros(nsteps)}
    for n in range(nsteps):
        Hy -= ch * (Ex[1:] - Ex[:-1])
        e1, e0, em2, em1 = Ex[1], Ex[0], Ex[-2], Ex[-1]
        Ex[1:-1] -= ce[1:-1] * (Hy[1:] - Hy[:-1])
        t = (n + 1) * DT
        Ex[KS] += np.exp(-((t - t0) / tw) ** 2) * np.sin(2 * np.pi * F0_1D * (t - t0))
        Ex[0] = e1 + mL * (Ex[1] - e0)
        Ex[-1] = em2 + mR * (Ex[-2] - em1)
        if pec_from is not None:
            Ex[pec_from:] = 0.0
        rec["p"][n], rec["t"][n] = Ex[KP], Ex[KT]
    return rec


def half_space(er2, kb=KB):
    er = np.ones(NZ)
    er[kb:] = er2
    er[kb] = (1 + er2) / 2          # the node on the boundary
    return er


t0 = time.time()
ref = fdtd1d(np.ones(NZ))           # air everywhere: the incident pulse alone
die = fdtd1d(half_space(4.0))       # air, then eps_r = 4 from KB on
print(f"two runs: {time.time() - t0:.2f} s")

steps = np.arange(4000)
tns = steps * DT * 1e9
inc = ref["p"]
refl = die["p"] - ref["p"]          # full run minus reference run: the reflected pulse alone
trans = die["t"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5.2), sharex=True)
ax1.plot(tns, die["p"], color="0.3")
ax1.set_ylabel(r"$E_x$ (V/m)")
ax1.set_title("At the probe, full run: incident and reflected pulses together")
ax2.plot(tns, inc, label="incident (reference run, at the probe)")
ax2.plot(tns, refl, label="reflected (full run − reference run, at the probe)")
ax2.plot(tns, trans, label="transmitted (in the dielectric)")
ax2.set_xlabel(r"$t$ (ns)"); ax2.set_ylabel(r"$E_x$ (V/m)"); ax2.set_xlim(0, 12)
for a in (ax1, ax2):
    a.grid(alpha=0.3)
ax2.legend(fontsize=9); plt.tight_layout(); plt.show()

### A1. $\Gamma$ and $\tau$ at normal incidence

The plot shows the incident pulse passing the probe, the reflected pulse coming
back past it, and the transmitted pulse in the dielectric. The full run alone
(top) holds the incident and reflected pulses together; subtracting the
reference run, with air everywhere, leaves the reflected pulse by itself.
Lesson 38 predicts, for air onto $\epsilon_r = 4$ ($\eta_2 = \eta_1/2$),

$$\Gamma = \frac{\eta_2 - \eta_1}{\eta_2 + \eta_1} = -\frac{1}{3}, \qquad \tau = 1 + \Gamma = \frac{2}{3}.$$

A pulse holds many frequencies, and the boundary treats each frequency in the
pulse separately: the Fourier transform of each pulse at $f_0 = 1\ \text{GHz}$
is its complex amplitude at $f_0$, and the ratio of two of them is the
coefficient at $f_0$, as if a single sinusoid had been sent. The reflected
pulse traveled $2(k_B - k_P)\Delta z$ farther than the incident pulse had when
it passed the probe, which adds a phase $e^{-j2\beta_1(k_B - k_P)\Delta z}$; the
transmitted pulse traveled $(k_B - k_P)\Delta z$ in air and
$(k_T - k_B)\Delta z$ in the dielectric. Undo those phases and the ratios are
$\Gamma$ and $\tau$ at the boundary.

**TODO:** compute `Gamma` and `tau` from the complex amplitudes below, then the
power check

$$\vert\Gamma\vert^2 + \frac{\eta_1}{\eta_2}\vert\tau\vert^2,$$

which Lesson 38's Poynting result says is 1.

In [ ]:
f = np.fft.rfftfreq(4000, DT)
i0 = np.argmin(abs(f - F0_1D))
I0, R0, T0 = (np.fft.rfft(s)[i0] for s in (inc, refl, trans))
beta1 = 2 * np.pi * F0_1D / C0
beta2 = 2 * np.pi * F0_1D * np.sqrt(4.0) / C0

Gamma = 0.0      # TODO: R0/I0 with the extra path's phase undone
tau = 0.0        # TODO: T0/I0 with both paths' phases undone
power = 0.0      # TODO: |Gamma|^2 + (eta1/eta2)|tau|^2

print(f"Gamma = {Gamma:.4f}: magnitude {abs(Gamma):.4f}, phase {np.degrees(np.angle(Gamma)):.2f} deg"
      f"   (Lesson 38: 0.3333 at 180 deg)")
print(f"tau   = {tau:.4f}: magnitude {abs(tau):.4f}, phase {np.degrees(np.angle(tau)):.2f} deg"
      f"   (Lesson 38: 0.6667 at 0 deg)")
print(f"power check = {power:.4f}")

**A1 questions.** (a) Give the error in the magnitude and in the phase of your
$\Gamma$ and of your $\tau$, against Lesson 38. (b) Which phase error is the
larger, and why? (Hint: the dielectric runs at $S = 1/2$, not $S = 1$; use
the dispersion relation above.) (c) The reflected pulse is upside down compared
with the incident pulse. Which part of $\Gamma$ says so?

**Answer:**

### A2. A perfect conductor

Lesson 38 showed that a perfect conductor reflects everything, with
$\Gamma = -1$. Set the field to zero from the boundary on and measure it.

**TODO:** compute `Gamma_pec` the same way as `Gamma`.

In [ ]:
pec = fdtd1d(np.ones(NZ), pec_from=KB)
R_pec = np.fft.rfft(pec["p"] - ref["p"])[i0]
Gamma_pec = 0.0    # TODO
print(f"Gamma_pec = {Gamma_pec:.4f}   (Lesson 38: -1)")

### A3. A quarter-wave anti-reflection layer

A camera lens, a solar cell, and a radome all carry a thin layer that cancels
the reflection at one frequency. A layer of index $n_c$ and thickness $d$
between air and $\epsilon_r = 4$ behaves like Lesson 7's section of
transmission line between a source line and a load: it transforms the
dielectric's $\eta_2$ into

$$\eta_\text{in} = \eta_c\ \frac{\eta_2 + j\eta_c\tan\beta_c d}{\eta_c + j\eta_2\tan\beta_c d}.$$

When $\beta_c d = \pi/2$ the tangent is infinite and
$\eta_\text{in} = \eta_c^2/\eta_2$, which equals $\eta_1$ when
$\eta_c = \sqrt{\eta_1\eta_2}$, or $n_c = \sqrt{n_1 n_2} = \sqrt{2}$. The layer is
a quarter wavelength thick inside itself: $d = \lambda_0/(4n_c)$.

**TODO:** set `n_c` and `d` for $f_0 = 1\ \text{GHz}$, and write the
theoretical $\vert\Gamma(f)\vert$ from $\eta_\text{in}$. The grid rounds $d$ to
a whole number of cells.

In [ ]:
n_c = 1.0          # TODO: index of the layer
d = 0.0            # TODO: thickness of the layer, m
nc_cells = int(round(d / DZ))
print(f"d = {d*1e3:.1f} mm = {d/DZ:.2f} cells -> {nc_cells} cells on the grid")

er = half_space(4.0)
er[KB - nc_cells:KB] = n_c ** 2
er[KB - nc_cells] = (1 + n_c ** 2) / 2
er[KB] = (n_c ** 2 + 4.0) / 2
coat = fdtd1d(er, nsteps=4000)

R_f = np.fft.rfft(coat["p"] - ref["p"], 16000)
I_f = np.fft.rfft(inc, 16000)
ff = np.fft.rfftfreq(16000, DT)
G_meas = abs(R_f / I_f)

eta1, eta2, eta_c = ETA0, ETA0 / 2, ETA0 / max(n_c, 1e-9)
bcd = 2 * np.pi * ff * n_c / C0 * nc_cells * DZ
G_theory = np.zeros_like(ff)  # TODO: |Gamma(f)| from eta_in

sel = (ff > 0.3e9) & (ff < 2.2e9)
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(ff[sel] / 1e9, G_meas[sel], label="FDTD")
ax.plot(ff[sel] / 1e9, G_theory[sel], "--", label="line theory (Lesson 7)")
ax.axhline(1/3, color="gray", lw=0.8, ls=":", label="no layer")
ax.set_xlabel(r"$f$ (GHz)"); ax.set_ylabel(r"$|\Gamma|$"); ax.legend(); ax.grid(alpha=0.3)
plt.show()
band = ff[sel][G_meas[sel] < 0.1]
if band.size:
    print(f"|Gamma| < 0.1 from {band.min()/1e9:.3f} to {band.max()/1e9:.3f} GHz")
for fx in (1e9, 2e9):
    i = np.argmin(abs(ff - fx))
    print(f"|Gamma| at {fx/1e9:.0f} GHz: FDTD {G_meas[i]:.4f}, theory {G_theory[i]:.4f}")

**A3 questions.** (a) Where is the null, and why is it not exactly at
1 GHz? (b) Over what band is $\vert\Gamma\vert < 0.1$? (c) At 2 GHz the same
layer is a half wavelength thick. What do the theory and the simulation give
for $\vert\Gamma\vert$ there, and why?

**Answer:**

---
## Part B: a beam in two dimensions

### The update equations

Lesson 39 put the boundary in the plane $z = 0$, medium 1 at $z < 0$, and the
plane of incidence in the $xz$ plane. Nothing changes along $y$, so
$\partial/\partial y = 0$, and the curl equations split into two independent
sets, one for each polarization of Lesson 39:

| | Fields | Faraday | Ampere–Maxwell |
| :-- | :-- | :-- | :-- |
| TE | $E_y,\ H_x,\ H_z$ | $\mu\ \partial H_x/\partial t = \partial E_y/\partial z$, $\ \mu\ \partial H_z/\partial t = -\partial E_y/\partial x$ | $\epsilon\ \partial E_y/\partial t = \partial H_x/\partial z - \partial H_z/\partial x$ |
| TM | $H_y,\ E_x,\ E_z$ | $\mu\ \partial H_y/\partial t = \partial E_z/\partial x - \partial E_x/\partial z$ | $\epsilon\ \partial E_x/\partial t = -\partial H_y/\partial z$, $\ \epsilon\ \partial E_z/\partial t = \partial H_y/\partial x$ |

Each line is one component of the curl: for TE, for example, the $y$ component
of $\nabla\times\mathbf{H}$ with $\partial/\partial y = 0$ is
$\partial H_x/\partial z - \partial H_z/\partial x$.

The grid staggers the TE set on a square grid of spacing $\Delta$: $E_y$ at
the nodes $(i, k)$, that is $x = i\Delta$ and $z = k\Delta$; $H_x$ halfway
along $z$, at $(i, k + \tfrac{1}{2})$; and $H_z$ halfway along $x$, at
$(i + \tfrac{1}{2}, k)$. The same centered differences as Part A give

$$H_x\big|^{n+1/2}_{i,k+1/2} = H_x\big|^{n-1/2}_{i,k+1/2} + \frac{\Delta t}{\mu\Delta}\left(E_y\big|^{n}_{i,k+1} - E_y\big|^{n}_{i,k}\right),$$

$$H_z\big|^{n+1/2}_{i+1/2,k} = H_z\big|^{n-1/2}_{i+1/2,k} - \frac{\Delta t}{\mu\Delta}\left(E_y\big|^{n}_{i+1,k} - E_y\big|^{n}_{i,k}\right),$$

$$E_y\big|^{n+1}_{i,k} = E_y\big|^{n}_{i,k} + \frac{\Delta t}{\epsilon\Delta}\left(H_x\big|^{n+1/2}_{i,k+1/2} - H_x\big|^{n+1/2}_{i,k-1/2} - H_z\big|^{n+1/2}_{i+1/2,k} + H_z\big|^{n+1/2}_{i-1/2,k}\right).$$

In the code, `F` is the field along $y$ ($E_y$ for TE), `A` is the in-plane
field halfway along $z$ ($H_x$), and `B` is the in-plane field halfway along
$x$ ($H_z$); `cF`, `cA`, and `cB` are their coefficients, $1/\epsilon$ or
$1/\mu$. The TM set runs through the same code with `F` $= H_y$, `A` $= E_x$,
and `B` $= E_z$, the coefficients exchanged, and the sign `s = -1`, which the
TM row of the table shows. The code stores `F` as `Fx + Fz`, the parts driven
by the $x$ and the $z$ derivatives, for the absorbing layer below.

**The node on the boundary.** The boundary falls on the row of nodes $k = k_B$.
For TE, $E_y$ sits on that row and is tangential, so it takes the average
permittivity, as in Part A. For TM, $E_x$ sits half a cell above or below the
row, wholly in one medium, but $E_z$ sits on the row and is *normal* to the
boundary. Its normal component is not continuous: $D_z = \epsilon E_z$ is
(Lesson 24). Faraday's law for the $H_y$ cell uses $E_z$ along the cell's side,
which crosses the boundary, so the grid needs the average of $E_z$ over that
side:

$$\frac{1}{\Delta}\int E_z\ dz = \frac{1}{\Delta}\int\frac{D_z}{\epsilon}\ dz = D_z\ \frac{1}{2}\left(\frac{1}{\epsilon_1} + \frac{1}{\epsilon_2}\right).$$

Since $\partial D_z/\partial t = \partial H_y/\partial x$ holds at the node, the
$E_z$ node on the boundary takes the average of $1/\epsilon$, not of
$\epsilon$. A tangential field averages $\epsilon$; a normal field averages
$1/\epsilon$.

**Time step.** Substituting $E_y = E_0e^{j(\omega n\Delta t - \beta_x i\Delta - \beta_z k\Delta)}$,
with amplitudes $X_0$ for $H_x$ and $Z_0$ for $H_z$, into the three TE updates,
and writing

$$\Omega = \frac{2}{\Delta t}\sin\frac{\omega\Delta t}{2}, \qquad K_x = \frac{2}{\Delta}\sin\frac{\beta_x\Delta}{2}, \qquad K_z = \frac{2}{\Delta}\sin\frac{\beta_z\Delta}{2},$$

gives

$$\mu\Omega X_0 = -K_zE_0, \qquad \mu\Omega Z_0 = K_xE_0, \qquad \epsilon\Omega E_0 = -K_zX_0 + K_xZ_0.$$

Substituting the first two into the third and dividing by $E_0$ gives
$\mu\epsilon\Omega^2 = K_x^2 + K_z^2$, or

$$\sin^2\frac{\omega\Delta t}{2} = S^2\left[\sin^2\frac{\beta_x\Delta}{2} + \sin^2\frac{\beta_z\Delta}{2}\right], \qquad S = \frac{v_p\Delta t}{\Delta}.$$

The worst case has both sines on the right equal to 1, which needs
$2S^2 \le 1$, so the Courant condition becomes $S \le 1/\sqrt{2}$. $S$ uses the
phase velocity of the medium: we set $S = 1/2$ in air, which makes it
$0.5/1.49 = 0.336$ in the acrylic, at 24 cells per free-space wavelength.
Unlike Part A, no grid in two dimensions is free of dispersion.

**Boundaries.** Part A's Mur boundary absorbs a wave that arrives head-on, but
a beam reaches the edges at many angles. Instead we line the grid with a
**perfectly matched layer (PML)**, 24 cells thick. Inside it each field obeys

$$\frac{\partial U}{\partial t} + rU = s,$$

where $s$ is the curl term and the loss rate $r$ grows smoothly toward the
outer edge. Multiplying by $e^{rt}$ turns the left side into one derivative,

$$\frac{d}{dt}\left(Ue^{rt}\right) = se^{rt},$$

and integrating over one step, with $s$ held constant, gives

$$U^{n+1}e^{r(t_n + \Delta t)} - U^ne^{rt_n} = \frac{s}{r}\left(e^{r(t_n + \Delta t)} - e^{rt_n}\right), \qquad U^{n+1} = e^{-r\Delta t}U^n + \frac{1 - e^{-r\Delta t}}{r}\ s.$$

The code splits each field into a part driven by the $x$ derivatives, which
loses energy at the rate $r_x$, and a part driven by the $z$ derivatives, at
the rate $r_z$. In a layer facing $z$ (top and bottom), only the part driven by
$\partial/\partial z$ decays, so the wave's variation along $x$, its phase
matching along the layer, keeps its form, and the layer stays matched to a wave
at any angle. Berenger showed that when $\mathbf{E}$ and $\mathbf{H}$ lose
energy at the same rate, such a layer reflects nothing at any angle in the
continuous limit; that proof is beyond this course, and on the grid the layer
reflects well under 1%.

**The source.** A line of cells parallel to the boundary drives the field with
$g(x)\sin(\omega t - \beta_1 x\sin\theta_i)$. The Gaussian taper $g(x)$ makes a
beam rather than an infinite plane wave, and the phase that advances along $x$
at the rate $\beta_1\sin\theta_i$ steers it to the angle $\theta_i$, which is
Lesson 39's phase-matching condition applied to aim the beam. The source line
also launches a beam upward, away from the boundary, which the absorbing layer
at the top removes.

**Measuring power.** After the field settles to a steady state, the code
extracts each field's phasor. For a field
$F = \text{Re}\big[\widetilde{F}e^{j\omega t}\big]$, multiplying by
$e^{-j\omega t}$ and averaging over a whole number of periods gives

$$\frac{1}{N}\sum F e^{-j\omega t} = \frac{1}{N}\sum\frac{\widetilde{F} + \widetilde{F}^*e^{-j2\omega t}}{2} = \frac{\widetilde{F}}{2},$$

because the $e^{-j2\omega t}$ term averages to zero; the code averages over four
periods of exactly 48 steps and doubles the result. It then integrates Lesson
38's time-average Poynting vector,
$\tfrac{1}{2}\text{Re}\big[\widetilde{\mathbf{E}}\times\widetilde{\mathbf{H}}^*\big]\cdot\hat{\mathbf{z}}$,
along a line parallel to the boundary. A second run with medium 1 everywhere
gives the incident field alone; subtracting it from the full run leaves the
reflected field. The code measures the reflected power above the boundary and
the transmitted power below it, each against the incident power: $R = \vert\Gamma\vert^2$,
and $T$ is Lesson 39's
$\vert\tau\vert^2(\eta_1\cos\theta_t)/(\eta_2\cos\theta_i)$. Each is measured on
its own, so $R + T = 1$ checks every run.

In [ ]:
F0 = 10e9                           # 10 GHz
LAM0 = C0 / F0                      # 3.0 cm
W0 = 2 * np.pi * F0
D = LAM0 / 24                       # 24 cells per free-space wavelength
DT2 = 0.5 * D / C0                  # S = 1/2 in air
NX, NZ2, NPML = 704, 240, 24
ZB, ZS, ZM, ZT = 100, 64, 82, 118   # boundary, source line, two measurement lines
X = (np.arange(NX) - NX // 2) * D   # x = 0 where the beam's center meets the boundary
Z = (np.arange(NZ2) - ZB) * D       # z = 0 on the boundary


def _pml_rate(n, rmax=3e11):
    s = np.arange(n) + 0.5
    p = np.zeros(n)
    p[:NPML] = ((NPML - s[:NPML]) / NPML) ** 3
    p[-NPML:] = ((s[-NPML:] - (n - NPML)) / NPML) ** 3
    return rmax * np.clip(p, 0, None)


_RX = _pml_rate(NX)[:, None] * np.ones((1, NZ2))
_RZ = _pml_rate(NZ2)[None, :] * np.ones((NX, 1))
_EX, _EZ = np.exp(-_RX * DT2), np.exp(-_RZ * DT2)
_GX = np.where(_RX > 0, (1 - _EX) / np.where(_RX > 0, _RX, 1), DT2)
_GZ = np.where(_RZ > 0, (1 - _EZ) / np.where(_RZ > 0, _RZ, 1), DT2)


def simulate(pol, n1, n2, theta_deg, w=2.5, periods=30, avg=4):
    # pol: "TE" (F = Ey, A = Hx, B = Hz) or "TM" (F = Hy, A = Ex, B = Ez).
    # Medium 1 (index n1) for z < 0. w is the beam's 1/e field half-width in
    # free-space wavelengths. Returns the phasors of F and of A on the grid.
    th = np.radians(theta_deg)
    eps = np.full((NX, NZ2), EPS0 * n1 ** 2)
    eps[:, ZB:] = EPS0 * n2 ** 2
    if pol == "TE":
        eps[:, ZB] = EPS0 * (n1 ** 2 + n2 ** 2) / 2     # tangential Ey on the boundary: average eps
        cF, cA, s, amp = 1 / eps, np.full_like(eps, 1 / MU0), 1.0, 1.0
        cB = cA
    else:
        cF, cA, s, amp = np.full_like(eps, 1 / MU0), 1 / eps, -1.0, 1.0 / ETA0
        cB = cA.copy()
        cB[:, ZB] = (1 / (EPS0 * n1 ** 2) + 1 / (EPS0 * n2 ** 2)) / 2   # normal Ez: average 1/eps
    wx = w * LAM0 / np.cos(th)                          # footprint of the beam on the source line
    x0 = -(ZB - ZS) * D * np.tan(th)                    # aim the center at x = 0 on the boundary
    g = np.exp(-((X - x0) / wx) ** 2)
    kx = n1 * W0 / C0 * np.sin(th)
    Fx, Fz, A, B = (np.zeros((NX, NZ2)) for _ in range(4))
    Fp = np.zeros((NX, NZ2), complex)
    Ap = np.zeros((NX, NZ2), complex)
    nsteps = int(periods / (F0 * DT2))
    nav = int(round(avg / (F0 * DT2)))                  # four whole periods of 48 steps
    for n in range(nsteps):
        t = n * DT2
        Fs = Fx + Fz
        A[:, :-1] = _EZ[:, :-1] * A[:, :-1] + s * _GZ[:, :-1] * cA[:, :-1] * (Fs[:, 1:] - Fs[:, :-1]) / D
        B[:-1, :] = _EX[:-1, :] * B[:-1, :] - s * _GX[:-1, :] * cB[:-1, :] * (Fs[1:, :] - Fs[:-1, :]) / D
        Fz[1:, 1:] = _EZ[1:, 1:] * Fz[1:, 1:] + s * _GZ[1:, 1:] * cF[1:, 1:] * (A[1:, 1:] - A[1:, :-1]) / D
        Fx[1:, 1:] = _EX[1:, 1:] * Fx[1:, 1:] - s * _GX[1:, 1:] * cF[1:, 1:] * (B[1:, 1:] - B[:-1, 1:]) / D
        Fz[:, ZS] += min(1.0, t * F0 / 3) * amp * g * np.sin(W0 * t - kx * X)
        if n >= nsteps - nav:                           # phasor: F = Re[F~ e^{j w t}]
            ph = np.exp(-1j * W0 * t)
            Fp += (Fx + Fz) * ph
            Ap += A * ph
    return 2 * Fp / nav, 2 * Ap / nav


def power_down(Fp, Ap, k, pol):
    # Power per meter (in y) crossing the line z = Z[k] in the +z direction.
    a = 0.5 * (Ap[:, k] + Ap[:, k - 1])                 # in-plane field at the same z
    if pol == "TE":
        Sz = -0.5 * np.real(Fp[:, k] * np.conj(a))      # S_z = -E_y H_x
    else:
        Sz = 0.5 * np.real(a * np.conj(Fp[:, k]))       # S_z =  E_x H_y
    return Sz.sum() * D


def reflect(pol, n1, n2, theta_deg):
    # Run the reference and the full simulation; return R, T, and both fields.
    Fr, Ar = simulate(pol, n1, n1, theta_deg)
    Ft, At = simulate(pol, n1, n2, theta_deg)
    P_i = power_down(Fr, Ar, ZM, pol)
    P_r = -power_down(Ft - Fr, At - Ar, ZM, pol)
    P_t = power_down(Ft, At, ZT, pol)
    return P_r / P_i, P_t / P_i, Ft, Fr


def show_field(ax, F, title, lim=None):
    # Re[F] with the boundary, the source line, the measurement lines, and the absorbing layer.
    lim = lim if lim is not None else np.abs(F).max()
    im = ax.imshow(np.real(F).T, extent=[X[0]/LAM0, X[-1]/LAM0, Z[-1]/LAM0, Z[0]/LAM0],
                   cmap="RdBu_r", vmin=-lim, vmax=lim, aspect="equal")
    ax.axhline(0, color="k", lw=1.0)
    ax.axhline(Z[ZS] / LAM0, color="C2", lw=1.2, label="source line")
    ax.axhline(Z[ZM] / LAM0, color="C1", lw=1.2, ls="--", label="power measured")
    ax.axhline(Z[ZT] / LAM0, color="C1", lw=1.2, ls="--")
    for z0, z1 in ((Z[0], Z[NPML]), (Z[-NPML], Z[-1])):
        ax.axhspan(z0 / LAM0, z1 / LAM0, color="0.6", alpha=0.35, lw=0)
    ax.set_xlim(-9, 11); ax.set_ylim(Z[-1] / LAM0, Z[0] / LAM0)
    ax.set_xlabel(r"$x/\lambda_0$"); ax.set_ylabel(r"$z/\lambda_0$")
    ax.set_title(title, fontsize=10)
    return im


t0 = time.time()
R45, T45, F45, F45ref = reflect("TE", 1.0, 1.49, 45.0)
print(f"TE at 45 deg, air onto n = 1.49: R = {R45:.4f}, T = {T45:.4f}, R + T = {R45+T45:.4f}"
      f"   ({time.time()-t0:.0f} s)")

fig, axs = plt.subplots(1, 2, figsize=(11, 4.0))
lim = np.abs(F45).max()
im = show_field(axs[0], F45, r"Full run: Re$[\tilde{E}_y]$, TE at 45° onto $n = 1.49$", lim)
F45r = F45 - F45ref
F45r[:, ZB:] = 0                    # below the boundary the difference has no meaning
show_field(axs[1], F45r, "Full run − reference run, above the boundary (own color scale)")
axs[0].legend(loc="lower left", fontsize=8)
fig.colorbar(im, ax=axs, label=r"Re$[\tilde{E}_y]$ (arb.)", shrink=0.8)
plt.show()

In both maps medium 1 (air) is on top and the acrylic below the black line;
the gray bands are the absorbing layer. The left map shows the incident and
reflected beams overlapping above the boundary and the transmitted beam below
it. The right map subtracts the reference run above the boundary, which leaves
only the reflected beam, drawn on its own, finer color scale; the code
integrates its power along the upper dashed line. Below the boundary the
difference means nothing, so the map leaves it blank.

### B1. Snell's law

The beam bends toward the normal as it enters the acrylic. Measure its angle:
find the beam's center (the centroid of $\vert\widetilde{E}_y\vert^2$ along $x$)
on two lines inside medium 2, and take the arctangent of the sideways shift over
the depth between them.

**TODO:** compute `theta_t_meas` and Lesson 39's Snell prediction
`theta_t_snell`.

In [ ]:
k1, k2 = ZB + 20, ZB + 100
I1, I2 = abs(F45[:, k1]) ** 2, abs(F45[:, k2]) ** 2
x1 = (X * I1).sum() / I1.sum()
x2 = (X * I2).sum() / I2.sum()
theta_t_meas = 0.0     # TODO: from x1, x2, and the depth (k2 - k1) * D
theta_t_snell = 0.0    # TODO: Snell's law, degrees
print(f"theta_t: measured {theta_t_meas:.2f} deg, Snell {theta_t_snell:.2f} deg")

fig, ax = plt.subplots(figsize=(6.5, 4.0))
show_field(ax, F45, "Beam centers in the acrylic, with the Snell ray")
ax.plot([x1 / LAM0, x2 / LAM0], [Z[k1] / LAM0, Z[k2] / LAM0], "ko", ms=6, label="measured centers")
zr = np.array([0, Z[k2]])
ax.plot(zr * np.tan(np.radians(theta_t_snell)) / LAM0, zr / LAM0, "k:", lw=1.5, label="Snell ray")
ax.legend(loc="lower left", fontsize=8); plt.show()

**B1 question.** Your measured angle differs from Snell's by a few tenths of a
degree. Why? (Hint: the next section explains that a beam is a sum of plane
waves over a spread of angles, and D1 explains the grid's phase velocity.)

**Answer:**

### B2. The Brewster angle

Lessons 39 and 40 derived, for nonmagnetic media,

$$\Gamma_\text{TE} = \frac{\eta_2\cos\theta_i - \eta_1\cos\theta_t}{\eta_2\cos\theta_i + \eta_1\cos\theta_t}, \qquad \Gamma_\text{TM} = \frac{\eta_2\cos\theta_t - \eta_1\cos\theta_i}{\eta_2\cos\theta_t + \eta_1\cos\theta_i},$$

and $\Gamma_\text{TM} = 0$ at the Brewster angle, $\tan\theta_B = n_2/n_1$, which
is $56.1^\circ$ from air onto acrylic.

**TODO:** write `fresnel(n1, n2, theta_deg)` to return
$(\Gamma_\text{TE}, \Gamma_\text{TM})$, then run the sweep. It runs ten pairs of
simulations, about two minutes.

In [ ]:
def fresnel(n1, n2, theta_deg):
    # TODO: return (Gamma_TE, Gamma_TM) from Lessons 39 and 40.
    th = np.radians(np.asarray(theta_deg, dtype=float))
    return 0 * th, 0 * th


angles = [0.0, 30.0, 45.0, 56.1, 65.0]
meas = {"TE": [], "TM": []}
t0 = time.time()
for th in angles:
    for pol in ("TE", "TM"):
        R, T, _, _ = reflect(pol, 1.0, 1.49, th)
        meas[pol].append(np.sqrt(max(R, 0.0)))
print(f"sweep: {time.time()-t0:.0f} s")

print(" angle   |Γ_TE| FDTD  theory   |Γ_TM| FDTD  theory")
for i, th in enumerate(angles):
    gte, gtm = fresnel(1.0, 1.49, th)
    print(f"{th:6.1f}   {meas['TE'][i]:8.4f} {abs(gte):8.4f}   {meas['TM'][i]:8.4f} {abs(gtm):8.4f}")

tt = np.linspace(0, 89, 400)
gte, gtm = fresnel(1.0, 1.49, tt)
thB = np.degrees(np.arctan(1.49))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(tt, abs(gte), label=r"$|\Gamma_\mathrm{TE}|$ theory")
ax.plot(tt, abs(gtm), label=r"$|\Gamma_\mathrm{TM}|$ theory")
ax.plot(angles, meas["TE"], "o", color="C0", label="TE, FDTD")
ax.plot(angles, meas["TM"], "s", color="C1", label="TM, FDTD")
ax.axvline(thB, color="gray", ls=":", lw=0.8, label=f"Brewster {thB:.1f}°")
ax.annotate("beam, not plane wave", (thB, meas["TM"][3]), xytext=(thB + 4, 0.08),
            fontsize=8, arrowprops=dict(arrowstyle="->", lw=0.8))
ax.set_xlabel(r"$\theta_i$ (degrees)"); ax.set_ylabel(r"$|\Gamma|$"); ax.legend(fontsize=8)
ax.grid(alpha=0.3); plt.show()

**B2 questions.** (a) At normal incidence TE and TM should agree. Do they, and
why might they differ slightly on a grid? (b) At $56.1^\circ$ the theory gives
$\vert\Gamma_\text{TM}\vert = 0$. What does the simulation give, and what
fraction of the incident power reflects for each polarization there?
(c) Explain why the measured $\vert\Gamma_\text{TM}\vert$ at Brewster is not
zero. (Hint: a beam is a sum of plane waves over a spread of angles of about
$\lambda/(\pi w)$ radians to each side, where $w = 2.5\lambda_0$ is the beam's
$1/e$ field half-width and $\lambda$ is the wavelength in medium 1.)

**Answer:**

### B3. Total internal reflection

Now reverse the media: the beam starts in acrylic and meets air at
$50^\circ$, beyond the critical angle $\theta_c = \arcsin(1/1.49) = 42.2^\circ$.
Lesson 40 predicts that all the power reflects and that the field in the air is
evanescent, decaying as $e^{-\alpha z}$ with

$$\alpha = \beta_2 q, \qquad q = \sqrt{\left(\frac{n_1}{n_2}\right)^2\sin^2\theta_i - 1}.$$

With medium 2 air, $n_2 = 1$ and $\beta_2 = \beta_0$.

**TODO:** fit a straight line to $\ln\vert\widetilde{E}_y\vert$ against $z$ just
inside the air, below the beam's center, to get the measured decay length
$1/\alpha$, and compute Lesson 40's value.

In [ ]:
t0 = time.time()
R_tir, T_tir, F_tir, _ = reflect("TE", 1.49, 1.0, 50.0)
print(f"acrylic to air at 50 deg: R = {R_tir:.4f}, T = {T_tir:.5f}   ({time.time()-t0:.0f} s)")

ix0 = np.argmin(abs(X))             # the column where the beam's center meets the boundary
zz = Z[ZB + 1:ZB + 10]
prof = abs(F_tir[ix0, ZB + 1:ZB + 10])
inv_alpha_meas = 0.0     # TODO: from a linear fit of log(prof) against zz, in units of LAM0
inv_alpha_theory = 0.0   # TODO: Lesson 40, in units of LAM0
print(f"1/alpha: measured {inv_alpha_meas:.3f} λ0, Lesson 40 {inv_alpha_theory:.3f} λ0")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={"width_ratios": [1.6, 1]})
im = show_field(ax1, F_tir, "Re$[\\tilde{E}_y]$: TE at 50° from acrylic (top) onto air (bottom)")
fig.colorbar(im, ax=ax1, label=r"Re$[\tilde{E}_y]$ (arb.)", shrink=0.8)
ax2.semilogy(zz / LAM0, prof / prof[0], "o", label="FDTD")
zf = np.linspace(0, zz[-1], 50)
if inv_alpha_meas > 0:
    ax2.semilogy(zf / LAM0, np.exp(-(zf - zz[0]) / (inv_alpha_meas * LAM0)), "-", label="fit")
if inv_alpha_theory > 0:
    ax2.semilogy(zf / LAM0, np.exp(-(zf - zz[0]) / (inv_alpha_theory * LAM0)), "--", label="Lesson 40")
ax2.set_xlabel(r"$z/\lambda_0$ (in the air)"); ax2.set_ylabel(r"$|\tilde{E}_y|$ (relative)")
ax2.set_title("Below the beam's center", fontsize=10)
ax2.legend(fontsize=8); ax2.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

**B3 questions.** (a) How close is $R$ to 1? (b) Compare your $1/\alpha$ with
Lesson 40's. Which way is the measured value off, and why? (Use the hint in
B2(c), with the wavelength in the acrylic, and remember that $\alpha$ shrinks as
$\theta_i$ falls toward $\theta_c$.) (c) In the field map, where does the field
in the air sit along $x$, and which way do its phase fronts run?

**Answer:**

---
## Discussion

**D1.** Part B uses 24 cells per free-space wavelength, which is 16 per
wavelength in the acrylic. Use the two-dimensional dispersion relation above,
for a wave along $x$ ($\beta_z = 0$), to find the grid's phase velocity in the
acrylic as a fraction of $v_p$. Remember that $S$ uses the acrylic's phase
velocity. How would doubling the resolution change it, and by what factor would
the run time grow?

**Answer:**

**D2.** Why does Part B measure power with the Poynting vector instead of
comparing field amplitudes, as Part A did?

**Answer:**

**D3.** Why must the beam be wide compared with a wavelength, and why can it
not be arbitrarily wide on this grid?

**Answer:**

**D4.** Name one device whose design depends on what you measured today, and
say which result it uses.

**Answer:**

---
## Report

Before you submit, run the whole notebook from the top (**Runtime**, then
**Run all** in Colab) so every output is current. Then:

1. Check that every `TODO` is done and every **Answer** cell is filled in.
2. Put every team member's name and your Documentation line in the first cell.
3. Export to PDF (**File**, then **Print**, then **Save as PDF**) and upload it to Gradescope as one team submission, adding every member.